In [0]:
catalog = "az_adb_simbus_training"
schema = "adarsh_training"
volume_path = "/Volumes/az_adb_simbus_training/adarsh_training/dataingestion"
# Using DBFS for checkpoints since checkpoints volume doesn't exist
checkpoint_path = "/dbfs/checkpoints/adarsh_training/dataingestion/"

datasets = ["CustomersRaw_Data", "LineitemRaw_Data", "OrdersRaw_Data", "NationRaw_Data"]

for table_name in datasets:
    df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("header", "true")
    .option("pathGlobFilter", f"*{table_name}*")
    .load(volume_path)
    )   
    (df.writeStream
    .option("checkpointLocation", f"{checkpoint_path}data_{table_name}")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.bronze_{table_name}")
    )  
    print(f"Auto Loader: bronze_{table_name} table is created.")